# SET Pharma — Google Colab Pipeline

This notebook contains the executable monthly SCMD feature-engineering and forecasting implementation. The repository supplies ingestion, reliability, RADCSR, simulation, validation and reporting.

**Research boundary:** Inventory outcomes are simulated. Stockout labels are not observed events.

Run the cells in order. Ingestion, model execution and mathematical validation are checkpointed as separate stages. The model stage remains a single long-running stage.

## 00. Runtime diagnostics

Inspect the allocated runtime before copying or processing the dataset. A GPU is optional; this workload may be limited by RAM, CPU and disk rather than GPU.

In [ ]:
import os, sys, shutil, subprocess, json, time
from pathlib import Path

print("Python:", sys.version)
if sys.version_info[:2] != (3, 13):
    raise RuntimeError(
        f"This Colab notebook requires Python 3.13.x; detected {sys.version.split()[0]}. "
        "Select a Python 3.13 Colab runtime and restart the session."
    )
print("Python 3.13 runtime verified.")
print("CPU cores:", os.cpu_count())
print("Disk free (GB):", round(shutil.disk_usage('/content').free / 1024**3, 2))
try:
    import psutil
    print("RAM total (GB):", round(psutil.virtual_memory().total / 1024**3, 2))
    print("RAM available (GB):", round(psutil.virtual_memory().available / 1024**3, 2))
except ImportError:
    print("psutil not installed yet")
try:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("PyTorch is not installed; GPU diagnostics skipped")

## 01. Clone the feature branch

The notebook deliberately checks out `feature/colab-pipeline-execution`, not `main`.

In [ ]:
REPO_URL = "https://github.com/Souptik-Hazra/SET_new.git"
BRANCH = "feature/colab-pipeline-execution"
REPO_DIR = Path("/content/SET_new")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR, check=True)

# Execute the reviewed snapshot, not the moving branch head.
subprocess.run(["git", "checkout", "--detach", f"origin/{BRANCH}"], cwd=REPO_DIR, check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print("Repository:", REPO_DIR)
print("Tracking branch:", BRANCH)
print("Detached checkout ensures the latest fetched branch commit is used.")
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR, text=True).strip())

## 02. Install project dependencies

Use the repository's pinned dependency ranges. Installation is performed in the active Colab runtime.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "psutil"], check=True)
print("Dependencies installed. If Colab requests a runtime restart, restart and continue from section 00.")

import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
print("Stage-level ingestion timings and DuckDB resource configuration will be logged below.")

## 03. Mount Google Drive and configure paths

Place the source CSV at the path below in Drive, or edit `DRIVE_RAW_CSV` to match its actual location. The raw file is copied to Colab's local disk for processing; it is never uploaded to Git.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_RAW_CSV = Path("/content/drive/MyDrive/Dataset/combined_secondary_care_medicines.csv")
DRIVE_ARTIFACT_ROOT = Path("/content/drive/MyDrive/SET_Pharma_Colab_Artifacts")
LOCAL_RAW_CSV = REPO_DIR / "data" / "raw" / "combined_secondary_care_medicines.csv"

if not DRIVE_RAW_CSV.is_file():
    raise FileNotFoundError(
        f"Upload the SCMD CSV to {DRIVE_RAW_CSV}, or update DRIVE_RAW_CSV."
    )

LOCAL_RAW_CSV.parent.mkdir(parents=True, exist_ok=True)
required_bytes = DRIVE_RAW_CSV.stat().st_size
free_bytes = shutil.disk_usage("/content").free
# Reserve room for the raw copy, aggregated Parquet, monthly Parquet and DuckDB spill files.
required_free_bytes = max(required_bytes * 3.0, required_bytes + 4 * 1024**3)
if free_bytes < required_free_bytes:
    raise RuntimeError(
        f"Insufficient Colab disk space: reserve {required_free_bytes / 1024**3:.2f} GB "
        f"(raw CSV + intermediate outputs + spill headroom); "
        f"available {free_bytes / 1024**3:.2f} GB. "
        "Free local runtime storage or use a runtime with more disk."
    )

if (
    not LOCAL_RAW_CSV.is_file()
    or LOCAL_RAW_CSV.stat().st_size != required_bytes
    or LOCAL_RAW_CSV.stat().st_mtime_ns != DRIVE_RAW_CSV.stat().st_mtime_ns
):
    print("Copying raw CSV from Drive to local Colab storage...")
    shutil.copy2(DRIVE_RAW_CSV, LOCAL_RAW_CSV)

print("Raw CSV:", LOCAL_RAW_CSV, "| GB:", round(LOCAL_RAW_CSV.stat().st_size / 1024**3, 3))
print("Persistent artifacts:", DRIVE_ARTIFACT_ROOT)


## 04. Execution controls

Keep the research defaults unless a documented experiment requires a change. Set `FORCE_* = True` only when intentionally rebuilding a stage. A checkpoint is reused only when its fingerprint matches and all expected outputs exist.

In [ ]:
from src.set_forecasting.config import get_config, validate_config

CONFIG = validate_config(get_config())
RUN_INGESTION = True
RUN_MODEL_PIPELINE = True
RUN_MATHEMATICAL_VALIDATION = True

FORCE_INGESTION = False
FORCE_MODEL_PIPELINE = False
FORCE_MATHEMATICAL_VALIDATION = False

RESTORE_CHECKPOINTS = False  # Set True only when intentionally resuming a previous run.

MIN_PERIODS = CONFIG["data"]["min_periods"]
MIN_NONZERO = CONFIG["data"]["min_nonzero"]
TRAIN_FRAC = CONFIG["data"]["train_frac"]
VALIDATION_FRAC = CONFIG["data"]["validation_frac"]
TEST_FRAC = CONFIG["data"]["test_frac"]
BOOTSTRAP_RESAMPLES = CONFIG["evaluation"]["bootstrap_resamples"]
RANDOM_SEED = CONFIG["seed"]

print(json.dumps({k: v for k, v in globals().copy().items() if k in [
    "RUN_INGESTION", "RUN_MODEL_PIPELINE", "RUN_MATHEMATICAL_VALIDATION",
    "FORCE_INGESTION", "FORCE_MODEL_PIPELINE", "FORCE_MATHEMATICAL_VALIDATION",
    "MIN_PERIODS", "MIN_NONZERO", "TRAIN_FRAC", "VALIDATION_FRAC", "TEST_FRAC", "BOOTSTRAP_RESAMPLES", "RANDOM_SEED"
]}, indent=2))


## 05. Restore previous checkpoints and import the runner

Only processed Parquet files and reports are restored from Drive. The raw CSV and repository are not copied back from the artifact folder.

In [ ]:
import importlib.util

required_imports = (
    "scripts.colab_runtime",
    "src.set_forecasting.config",
    "src.set_forecasting.pipeline",
    "src.visualization.generate_figures",
)
missing_imports = [name for name in required_imports if importlib.util.find_spec(name) is None]
if missing_imports:
    raise ImportError(f"Repository modules are not importable: {missing_imports}")
print("Repository import checks passed:", ", ".join(required_imports))

from scripts.colab_runtime import (
    environment_report, sync_artifacts, run_ingestion,
    run_models, run_mathematical_validation, DISCLAIMER
)

print(json.dumps(environment_report(), indent=2))
if RESTORE_CHECKPOINTS:
    restored = sync_artifacts(DRIVE_ARTIFACT_ROOT, REPO_DIR) if DRIVE_ARTIFACT_ROOT.exists() else []
    print("Restored artifact files:", len(restored))
else:
    for relative in ("data/processed", "output"):
        target = REPO_DIR / relative
        if target.exists():
            shutil.rmtree(target)
        target.mkdir(parents=True, exist_ok=True)
    restored = []
    print("Fresh run selected: cleared local processed artifacts, reports and checkpoint manifest.")
print(DISCLAIMER)

## 06. Phase 1 — SCMD ingestion

DuckDB streams and aggregates the CSV. The monthly cohort is written to Parquet. The raw file remains outside Git. After success, processed outputs and the stage manifest are copied to Drive.

In [ ]:
if RUN_INGESTION:
    ingestion_result = run_ingestion(
        LOCAL_RAW_CSV,
        min_periods=MIN_PERIODS,
        min_nonzero=MIN_NONZERO,
        train_frac=TRAIN_FRAC,
        val_frac=VALIDATION_FRAC,
        test_frac=TEST_FRAC,
        force=FORCE_INGESTION,
    )
    print(ingestion_result)
    saved = sync_artifacts(REPO_DIR, DRIVE_ARTIFACT_ROOT)
    print("Persisted files to Drive:", len(saved))
else:
    print("Ingestion disabled by configuration")

### 06A. Ingestion output preflight

Check that the SQL ingestion stage produced a readable, non-empty Parquet file with the exact columns required by the forecasting pipeline before starting the longer model stage.

In [ ]:
import pyarrow.parquet as pq

SERIES_PARQUET = REPO_DIR / "data" / "processed" / "scmd_monthly_series.parquet"
required_series_columns = {
    "ODS_CODE",
    "VMP_SNOMED_CODE",
    "UNIT_OF_MEASURE_IDENTIFIER",
    "PERIOD",
    "MODEL_DEMAND",
    "OBSERVATION_MISSING",
    "SPLIT",
}

if not SERIES_PARQUET.is_file() or SERIES_PARQUET.stat().st_size == 0:
    raise RuntimeError(
        "Ingestion did not create a non-empty monthly series Parquet file. "
        "Review the Phase 1 error and output/phase1_diagnostics.json."
    )

parquet_metadata = pq.ParquetFile(SERIES_PARQUET)
available_columns = set(parquet_metadata.schema.names)
missing_columns = sorted(required_series_columns - available_columns)
if missing_columns:
    raise RuntimeError(
        f"Monthly series Parquet is missing required columns: {missing_columns}"
    )
if parquet_metadata.metadata.num_rows == 0:
    raise RuntimeError("Monthly series Parquet contains zero rows.")

print("Ingestion output preflight passed.")
print("Monthly rows:", f"{parquet_metadata.metadata.num_rows:,}")
print("Parquet size (GB):", round(SERIES_PARQUET.stat().st_size / 1024**3, 3))
print("Required schema columns:", len(required_series_columns))

try:
    import psutil

    available_ram_gb = psutil.virtual_memory().available / 1024**3
    print("RAM currently available (GB):", round(available_ram_gb, 2))
    estimated_working_set_gb = max(6.0, (SERIES_PARQUET.stat().st_size / 1024**3) * 6.0)
    print("Estimated minimum forecasting working-set (GB):", round(estimated_working_set_gb, 2))
    if available_ram_gb < estimated_working_set_gb:
        raise RuntimeError(
            "Forecasting stopped before model execution: available RAM is below the "
            "conservative working-set estimate. Reconnect with a higher-RAM Colab runtime "
            "or optimise the pandas forecasting stage before proceeding."
        )
    print("RAM preflight passed against the conservative working-set estimate.")
except ImportError:
    print("Install psutil to inspect available RAM before forecasting.")


## 07. Notebook-owned forecasting implementation

The complete memory-optimised monthly SCMD feature engineering and quantile forecasting implementation is embedded in this notebook and kept aligned with the corresponding version-controlled source modules. It uses vectorised grouped features, train-only per-series winsorisation, Q10/Q50/Q90 LightGBM estimators and previous-observation/seasonal-naive baselines.

Reliability estimation, RADCSR formulas and coefficients, scenario simulation, calibration, bootstrap inference and mathematical validation remain in their canonical research modules. Inventory outcomes and stockout labels remain simulated.

In [ ]:
# Notebook-owned memory-optimised forecasting implementation, mirrored in src/.
"""Vectorised past-only SCMD feature engineering."""
import numpy as np
import pandas as pd

SERIES_KEYS = ["ODS_CODE", "VMP_SNOMED_CODE", "UNIT_OF_MEASURE_IDENTIFIER"]
TARGET = "MODEL_DEMAND"
LAGS = (1, 2, 3, 6, 12)
WINDOWS = (3, 6, 12)


def _rolling(grouped, column, window, operation):
    values = getattr(grouped[column].rolling(window, min_periods=1), operation)()
    return values.reset_index(level=list(range(len(SERIES_KEYS))), drop=True).sort_index()


def create_features(data):
    """Build all lag/rolling features without a Python loop over 389k+ series."""
    required = set(SERIES_KEYS + ["PERIOD", TARGET, "OBSERVATION_MISSING"])
    missing = required - set(data.columns)
    if missing:
        raise ValueError(f"Missing SCMD columns: {sorted(missing)}")
    if data.empty:
        return data.copy()

    g = data.copy()
    g["PERIOD"] = pd.to_datetime(g["PERIOD"], errors="raise")
    g = g.sort_values(SERIES_KEYS + ["PERIOD"], kind="mergesort").reset_index(drop=True)
    grouped = g.groupby(SERIES_KEYS, sort=False, dropna=False, observed=True)
    missing_obs = g["OBSERVATION_MISSING"].fillna(True).astype(bool)
    y = pd.to_numeric(g[TARGET], errors="coerce").where(~missing_obs)
    g["_Y"] = y
    g["_PAST"] = grouped["_Y"].shift(1)
    grouped = g.groupby(SERIES_KEYS, sort=False, dropna=False, observed=True)

    for lag in LAGS:
        g[f"LAG_{lag}"] = grouped["_Y"].shift(lag)

    for window in WINDOWS:
        mean = _rolling(grouped, "_PAST", window, "mean")
        median = _rolling(grouped, "_PAST", window, "median")
        std = _rolling(grouped, "_PAST", window, "std")
        minimum = _rolling(grouped, "_PAST", window, "min")
        maximum = _rolling(grouped, "_PAST", window, "max")
        count = _rolling(grouped, "_PAST", window, "count")
        zeros = g["_PAST"].eq(0).astype("float64")
        g["_ZERO"] = zeros
        grouped_zero = g.groupby(SERIES_KEYS, sort=False, dropna=False, observed=True)
        zero_count = _rolling(grouped_zero, "_ZERO", window, "sum")
        g[f"ROLL_MEAN_{window}"] = mean
        g[f"ROLL_MEDIAN_{window}"] = median
        g[f"ROLL_STD_{window}"] = std.fillna(0.0)
        g[f"ROLL_MIN_{window}"] = minimum
        g[f"ROLL_MAX_{window}"] = maximum
        g[f"ROLL_CV_{window}"] = g[f"ROLL_STD_{window}"] / mean.replace(0, np.nan)
        g[f"ZERO_RATIO_{window}"] = zero_count / count.replace(0, np.nan)
        g["_OBSERVED"] = (~missing_obs).astype("float64")
        g["_OBSERVED_PAST"] = g.groupby(
            SERIES_KEYS, sort=False, dropna=False, observed=True
        )["_OBSERVED"].shift(1)
        observed_grouped = g.groupby(SERIES_KEYS, sort=False, dropna=False, observed=True)
        g[f"OBSERVED_RATIO_{window}"] = _rolling(
            observed_grouped.assign if False else observed_grouped,
            "_OBSERVED_PAST", window, "mean"
        )

    row_number = g.groupby(SERIES_KEYS, sort=False, dropna=False, observed=True).cumcount()
    last_nonzero = row_number.where(g["_PAST"].gt(0)).groupby(
        [g[key] for key in SERIES_KEYS], sort=False, dropna=False
    ).ffill()
    g["MONTHS_SINCE_NONZERO"] = row_number - last_nonzero
    g["_POSITIVE_PAST"] = g["_PAST"].where(g["_PAST"] > 0)
    positive_grouped = g.groupby(SERIES_KEYS, sort=False, dropna=False, observed=True)
    g["PRIOR_NONZERO_MEAN_6"] = _rolling(positive_grouped, "_POSITIVE_PAST", 6, "mean")
    g["MOMENTUM_3M"] = g["ROLL_MEAN_3"] / g["ROLL_MEAN_6"].replace(0, np.nan)
    g["SEASONAL_RATIO_12"] = g["LAG_12"] / g["ROLL_MEAN_12"].replace(0, np.nan)

    month = g["PERIOD"].dt.month
    g["MONTH"] = month
    g["QUARTER"] = g["PERIOD"].dt.quarter
    g["YEAR"] = g["PERIOD"].dt.year
    g["MONTH_SIN"] = np.sin(2 * np.pi * month / 12)
    g["MONTH_COS"] = np.cos(2 * np.pi * month / 12)
    g["TREND_INDEX"] = row_number.astype("int32")
    g["TARGET_OBSERVED"] = ~missing_obs

    if {"TOTAL_INDICATIVE_COST", "POSITIVE_QUANTITY"}.issubset(g.columns):
        cost = pd.to_numeric(g["TOTAL_INDICATIVE_COST"], errors="coerce")
        quantity = pd.to_numeric(g["POSITIVE_QUANTITY"], errors="coerce")
        unit_cost = (cost / quantity.where(quantity > 0)).replace([np.inf, -np.inf], np.nan)
        g["_UNIT_COST"] = unit_cost
        unit_grouped = g.groupby(SERIES_KEYS, sort=False, dropna=False, observed=True)
        lagged_cost = unit_grouped["_UNIT_COST"].shift(1)
        g["_LAGGED_UNIT_COST"] = lagged_cost
        cost_grouped = g.groupby(SERIES_KEYS, sort=False, dropna=False, observed=True)
        g["LAG_UNIT_COST_1"] = lagged_cost
        g["ROLL_UNIT_COST_3"] = _rolling(cost_grouped, "_LAGGED_UNIT_COST", 3, "mean")

    return g.drop(columns=[
        "_Y", "_PAST", "_ZERO", "_OBSERVED", "_OBSERVED_PAST",
        "_POSITIVE_PAST", "_UNIT_COST", "_LAGGED_UNIT_COST"
    ], errors="ignore")


def leakage_check(features):
    required = set(SERIES_KEYS + ["PERIOD", TARGET, "OBSERVATION_MISSING", "LAG_1"])
    if required - set(features.columns):
        return False
    ordered = features.sort_values(SERIES_KEYS + ["PERIOD"], kind="mergesort")
    grouped = ordered.groupby(SERIES_KEYS, sort=False, dropna=False, observed=True)
    expected = grouped[TARGET].shift(1).where(
        ~grouped["OBSERVATION_MISSING"].shift(1).fillna(True).astype(bool)
    )
    actual = pd.to_numeric(ordered["LAG_1"], errors="coerce")
    return bool(np.allclose(actual.to_numpy(float), expected.to_numpy(float), equal_nan=True))


"""Memory-conscious global quantile LightGBM and naive forecasting baselines."""
import os
import numpy as np
import pandas as pd
from ..features.engineering import create_features, SERIES_KEYS, TARGET

FEATURES = [
    "LAG_1", "LAG_2", "LAG_3", "LAG_6", "LAG_12",
    "ROLL_MEAN_3", "ROLL_MEDIAN_3", "ROLL_STD_3", "ROLL_CV_3",
    "ROLL_MEAN_6", "ROLL_MEDIAN_6", "ROLL_STD_6", "ROLL_CV_6",
    "ROLL_MIN_6", "ROLL_MAX_6", "ZERO_RATIO_6", "OBSERVED_RATIO_6",
    "MONTHS_SINCE_NONZERO", "PRIOR_NONZERO_MEAN_6", "MOMENTUM_3M",
    "SEASONAL_RATIO_12", "MONTH", "QUARTER", "YEAR", "MONTH_SIN",
    "MONTH_COS", "TREND_INDEX", "LAG_UNIT_COST_1", "ROLL_UNIT_COST_3",
]


def _score(frame, name):
    y = frame["MODEL_DEMAND"].to_numpy(float)
    low = frame["Q10"].to_numpy(float)
    median = frame["Q50"].to_numpy(float)
    high = frame["Q90"].to_numpy(float)
    valid = np.isfinite(y) & np.isfinite(low) & np.isfinite(median) & np.isfinite(high)
    y, low, median, high = y[valid], low[valid], median[valid], high[valid]
    if not len(y):
        return {"MODEL": name, "N": 0}
    pinball = lambda q, p: float(np.mean(np.maximum(q * (y - p), (q - 1) * (y - p))))
    return {
        "MODEL": name, "N": len(y),
        "MAE": float(np.mean(np.abs(y - median))),
        "WAPE": float(np.abs(y - median).sum() / max(np.abs(y).sum(), 1e-9) * 100),
        "PICP_80": float(np.mean((y >= low) & (y <= high))),
        "MPIW_80": float(np.mean(high - low)),
        "PINBALL_10": pinball(.1, low),
        "PINBALL_50": pinball(.5, median),
        "PINBALL_90": pinball(.9, high),
    }


def _winsorise_by_series(frame, quantile):
    """Calculate train-only positive-demand thresholds with one grouped aggregation."""
    train_mask = frame["SPLIT"].eq("train") & ~frame["OBSERVATION_MISSING"].fillna(True).astype(bool)
    positive_train = pd.to_numeric(frame.loc[train_mask, "MODEL_DEMAND"], errors="coerce")
    positive_train = positive_train[positive_train > 0]
    keys = frame.loc[positive_train.index, SERIES_KEYS]
    thresholds = positive_train.groupby(
        [keys[column] for column in SERIES_KEYS], sort=False, dropna=False
    ).quantile(quantile)
    row_keys = pd.MultiIndex.from_frame(frame[SERIES_KEYS])
    limits = thresholds.reindex(row_keys).to_numpy(dtype="float64")
    limits[~np.isfinite(limits)] = np.inf
    values = pd.to_numeric(frame["MODEL_DEMAND"], errors="coerce").to_numpy(dtype="float64")
    winsorised = np.where(values > 0, np.minimum(values, limits), values)
    return limits, winsorised, values > limits


def train_and_predict(series, min_train_rows=100, random_state=42, winsor_quantile=.995):
    """Train quantile models with bounded thread use and avoid per-series Python loops."""
    required = set(SERIES_KEYS + ["PERIOD", "MODEL_DEMAND", "OBSERVATION_MISSING", "SPLIT"])
    missing = required - set(series.columns)
    if missing:
        raise ValueError(f"Monthly series is missing columns: {sorted(missing)}")

    # Keep only source columns needed downstream; avoid an extra full-frame copy.
    keep = list(dict.fromkeys(
        SERIES_KEYS + [
            "PERIOD", "MODEL_DEMAND", "OBSERVATION_MISSING", "SPLIT",
            "YEAR_MONTH", "TRAIN_END", "VALIDATION_END", "TEST_START",
            "TOTAL_INDICATIVE_COST", "POSITIVE_QUANTITY", "NEGATIVE_QUANTITY",
            "NET_QUANTITY", "SOURCE_RECORDS", "VALID_QUANTITY_RECORDS",
            "INVALID_QUANTITY_RECORDS", "MISSING_OR_INVALID_COST_RECORDS",
            "NEGATIVE_COST_RECORDS", "COHORT_INCLUDED",
        ]
    ))
    x = series.loc[:, [column for column in keep if column in series.columns]].copy()
    x["PERIOD"] = pd.to_datetime(x["PERIOD"], errors="raise")
    original = pd.to_numeric(x["MODEL_DEMAND"], errors="coerce")
    limits, winsorised, outliers = _winsorise_by_series(x, winsor_quantile)
    x["MODEL_DEMAND_WINSORIZED"] = winsorised
    x["OUTLIER_995_FLAG"] = outliers
    x["WINSOR_LIMIT_995"] = limits
    x["_ORIGINAL_MODEL_DEMAND"] = original
    x[TARGET] = x["MODEL_DEMAND_WINSORIZED"]

    features = create_features(x)
    features["MODEL_DEMAND"] = features["_ORIGINAL_MODEL_DEMAND"]
    features["SERIES_ID"] = features[SERIES_KEYS].astype(str).agg("|".join, axis=1)
    categorical = []
    for column in SERIES_KEYS:
        name = column + "_CAT"
        features[name] = features[column].astype("category")
        categorical.append(name)
    columns = [column for column in FEATURES if column in features.columns] + categorical
    for column in FEATURES:
        if column in features:
            features[column] = pd.to_numeric(features[column], errors="coerce")

    train_mask = features["SPLIT"].eq("train") & features["TARGET_OBSERVED"]
    val_mask = features["SPLIT"].eq("validation") & features["TARGET_OBSERVED"]
    validation_mask = features["SPLIT"].eq("validation")
    test_mask = features["SPLIT"].eq("test")
    train = features.loc[train_mask & features["MODEL_DEMAND_WINSORIZED"].notna()]
    val = features.loc[val_mask & features["MODEL_DEMAND_WINSORIZED"].notna()]
    validation_calendar = features.loc[validation_mask]
    test = features.loc[test_mask]

    models = []
    status = "naive_fallback"
    try:
        import lightgbm as lgb
        from lightgbm import LGBMRegressor

        if len(train) >= min_train_rows and len(val) >= 20:
            available_cpus = os.cpu_count() or 2
            model_threads = max(1, min(4, available_cpus))
            train_x = train[columns]
            train_y = train["MODEL_DEMAND_WINSORIZED"]
            val_x = val[columns]
            val_y = val["MODEL_DEMAND_WINSORIZED"]
            for quantile in (.1,.5,.9):
                estimator = LGBMRegressor(
                    objective="quantile", alpha=quantile, n_estimators=800,
                    learning_rate=.03, num_leaves=31, min_child_samples=30,
                    subsample=.85, colsample_bytree=.85, max_bin=127,
                    force_col_wise=True, verbosity=-1, random_state=random_state,
                    n_jobs=model_threads,
                )
                estimator.fit(
                    train_x, train_y, categorical_feature=categorical,
                    eval_set=[(val_x, val_y)],
                    callbacks=[lgb.early_stopping(50, verbose=False)],
                )
                models.append(estimator)
            status = "quantile_lightgbm"
    except ImportError:
        pass

    def predict(part):
        result = part.copy()
        if result.empty:
            return result
        if models:
            raw = np.column_stack([
                np.maximum(0, estimator.predict(result[columns])) for estimator in models
            ])
            result[["Q10", "Q50", "Q90"]] = np.sort(raw, axis=1)
        else:
            result["Q50"] = result["LAG_1"].fillna(result["LAG_12"]).fillna(0).clip(lower=0)
            result["Q10"] = result["Q50"] * .5
            result["Q90"] = result["Q50"] * 1.5
        result["FORECAST_MODEL"] = status
        result["NAIVE_LAST"] = result["LAG_1"].fillna(0).clip(lower=0)
        result["NAIVE_SEASONAL_12"] = result["LAG_12"].fillna(result["NAIVE_LAST"]).clip(lower=0)
        return result

    validation_pred = predict(validation_calendar)
    test_pred = predict(test)
    all_predictions = pd.concat([validation_pred, test_pred], ignore_index=True)
    metrics = []
    if len(test_pred):
        metrics.append(_score(
            test_pred, "QuantileLightGBM" if status == "quantile_lightgbm" else "NaiveFallback"
        ))
        for column, name in (
            ("NAIVE_LAST", "PreviousObservationNaive"),
            ("NAIVE_SEASONAL_12", "SeasonalNaive12"),
        ):
            baseline = test_pred[["MODEL_DEMAND", column]].copy()
            baseline["Q50"] = baseline[column]
            baseline["Q10"] = baseline[column] * .5
            baseline["Q90"] = baseline[column] * 1.5
            metrics.append(_score(baseline, name))

    report = {
        "model": status, "train_rows": len(train), "validation_rows": len(val),
        "validation_calendar_rows": len(validation_calendar), "test_rows": len(test_pred),
        "test_observed_rows": int(test_pred["TARGET_OBSERVED"].sum()),
        "validation_predictions": len(validation_pred),
        "series_count": int(features.groupby(SERIES_KEYS, sort=False, dropna=False).ngroups),
        "feature_rows": int(len(features)),
        "lightgbm_threads": max(1, min(4, os.cpu_count() or 2)),
    }
    return features, all_predictions, pd.DataFrame(metrics), report


import src.set_forecasting.pipeline as pipeline_module
pipeline_module.train_and_predict = train_and_predict
print("Notebook-owned forecasting implementation installed: vectorised grouped features, grouped winsorisation, bounded LightGBM threads.")


## 08. Forecasting, reliability, RADCSR and simulation

The downstream pipeline consumes the notebook-defined `train_and_predict()` implementation, then runs reliability estimation, RADCSR, scenario simulation, calibration and statistical evaluation through the existing research modules. This remains one long-running model stage; a disconnect requires rerunning this stage, while a completed ingestion checkpoint can be reused.

In [ ]:
if RUN_MODEL_PIPELINE:
    import gc
    import time
    import psutil

    gc.collect()
    memory = psutil.virtual_memory()
    print(
        f"Model stage starting | RAM available: {memory.available / 1024**3:.2f} GB "
        f"| RAM used: {memory.percent:.1f}%",
        flush=True,
    )
    model_started = time.monotonic()
    try:
        # Keep native numerical libraries from oversubscribing Colab CPU/RAM.
        try:
            from threadpoolctl import threadpool_limits
            thread_limit = threadpool_limits(limits=4)
        except ImportError:
            thread_limit = None

        model_result = run_models(
            LOCAL_RAW_CSV,
            min_periods=MIN_PERIODS,
            min_nonzero=MIN_NONZERO,
            train_frac=TRAIN_FRAC,
            val_frac=VALIDATION_FRAC,
            test_frac=TEST_FRAC,
            bootstrap=BOOTSTRAP_RESAMPLES,
            seed=RANDOM_SEED,
            force=FORCE_MODEL_PIPELINE,
        )
        print(model_result)
    except Exception:
        import traceback
        traceback.print_exc()
        raise
    finally:
        elapsed = time.monotonic() - model_started
        gc.collect()
        memory = psutil.virtual_memory()
        print(
            f"Model stage elapsed: {elapsed / 60:.1f} min | "
            f"RAM available after stage: {memory.available / 1024**3:.2f} GB",
            flush=True,
        )
        saved = sync_artifacts(REPO_DIR, DRIVE_ARTIFACT_ROOT)
        print("Persisted files to Drive:", len(saved), flush=True)
else:
    print("Model pipeline disabled by configuration")


## 10. Phase B mathematical validation

This requires the forecast and simulation artifacts from the model stage. It evaluates the current implementation and writes JSON and Markdown reports.

In [ ]:
if RUN_MATHEMATICAL_VALIDATION:
    math_result = run_mathematical_validation(
        bootstrap=BOOTSTRAP_RESAMPLES,
        seed=RANDOM_SEED,
        force=FORCE_MATHEMATICAL_VALIDATION,
    )
    print("Mathematical validation status:", math_result["status"])
    saved = sync_artifacts(REPO_DIR, DRIVE_ARTIFACT_ROOT)
    print("Persisted files to Drive:", len(saved))
else:
    print("Mathematical validation disabled by configuration")

## 11. Generate, save and display research figures

Figures are generated from the pipeline's saved Parquet and CSV artifacts. The generator does not insert illustrative metric values. Figures based on synthetic inventory outcomes are explicitly labelled as simulated. Each PNG is saved under `output/figures/`, displayed below, and synchronized to Google Drive.

In [ ]:
from IPython.display import Image, display
from src.visualization.generate_figures import generate_figures

generated_figures = generate_figures(REPO_DIR)
if generated_figures:
    for figure_path in generated_figures:
        print(figure_path.name)
        display(Image(filename=str(figure_path), width=1000))
else:
    print("No figures were generated. Inspect output/figures/figure_generation_report.json for missing inputs.")

saved = sync_artifacts(REPO_DIR, DRIVE_ARTIFACT_ROOT)
print("Persisted figures and reports to Drive:", len(saved))
print(DISCLAIMER)

## 12. Review generated outputs

Inspect reports before drawing research conclusions. Simulated stockout labels are not observed hospital stockout events.

In [ ]:
report_paths = [
    REPO_DIR / "output" / "pipeline_report.json",
    REPO_DIR / "output" / "mathematical_validation.json",
    REPO_DIR / "output" / "final_verdict.md",
    REPO_DIR / "output" / "figures" / "figure_generation_report.json",
]
for path in report_paths:
    print(("FOUND  " if path.is_file() else "MISSING ") + str(path))
    if path.is_file(): print("  Size:", path.stat().st_size, "bytes")

processed = REPO_DIR / "data" / "processed"
print("\nProcessed artifacts:")
for path in sorted(processed.glob("*")):
    if path.is_file(): print(f"{path.name:45s} {path.stat().st_size / 1024**2:8.2f} MB")

figures_dir = REPO_DIR / "output" / "figures"
print("\nSaved figures:")
for path in sorted(figures_dir.glob("*.png")):
    print(f"{path.name:48s} {path.stat().st_size / 1024:.1f} KB")

print("\n", DISCLAIMER)

## 13. Re-running selected stages

- To rerun only mathematical validation, set `RUN_INGESTION = False`, `RUN_MODEL_PIPELINE = False`, `RUN_MATHEMATICAL_VALIDATION = True` and rerun the relevant cells.
- To rebuild forecasts and downstream outputs from an existing monthly series, disable ingestion and force the model stage.
- To rebuild the monthly cohort, force ingestion; downstream stages should then be rerun.
- If the runtime disconnects, reconnect, rerun setup and Drive restore cells, then rerun the desired stage cells.

The checkpoint fingerprint includes source-file metadata, hashes of the code files used by that stage, the configuration hash and stage parameters. The Git revision is recorded in the manifest for audit. Changing relevant implementation code or configuration invalidates the affected checkpoint.